In [1]:
!mkdir -p /content/DIV2K

# Valid HR 데이터 다운로드 및 압축 해제
!wget http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip -O /content/DIV2K/valid_hr.zip
!unzip -q /content/DIV2K/valid_hr.zip -d /content/DIV2K
!rm /content/DIV2K/valid_hr.zip

print("데이터셋 다운로드 완료")

--2026-02-25 11:43:08--  http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip
Resolving data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)... 129.132.52.178, 2001:67c:10ec:36c2::178
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip [following]
--2026-02-25 11:43:09--  https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 448993893 (428M) [application/zip]
Saving to: ‘/content/DIV2K/valid_hr.zip’

/content/DIV2K/vali 100%[===================>] 428.19M  20.3MB/s    in 22s     

2026-02-25 11:43:32 (19.2 MB/s) - ‘/content/DIV2K/valid_hr.zip’ saved [448993893/448993893]

✅ 데이터셋 다운로드 및 준비 완료!


In [2]:
import os, glob, math, random
import numpy as np
from PIL import Image
import hashlib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from tqdm.auto import tqdm

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# ============================================================
# 0) Reproducibility
# ============================================================
SEED = 614
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

DEVICE: cuda


##SRNO

In [4]:
# ============================================================
# 1) EXP DIRS
# ============================================================
EXP_DIRS = {
    "baseline":  "/content/drive/MyDrive/SRNO_Base_final",
    "fourier5":  "/content/drive/MyDrive/SRNO_Fourier5_final",
    "fourier10": "/content/drive/MyDrive/SRNO_Fourier10_final",
    "inv_area":  "/content/drive/MyDrive/SRNO_LocalAgg_final",
}

In [5]:
# ============================================================
# 2) DIV2K valid files
# ============================================================
DIV2K_ROOT = "/content/DIV2K"
VALID_GLOB = os.path.join(DIV2K_ROOT, "DIV2K_valid_HR", "*.png")
valid_files = sorted(glob.glob(VALID_GLOB))
assert len(valid_files) == 100, f"Expected 100 valid images, got {len(valid_files)}"
print("Valid images:", len(valid_files))

Valid images: 100


In [6]:
# ============================================================
# 3) Utils (same as srno_exp_base_final.py / srno_comparison.py)
# ============================================================
def norm_01_to_m11(x):
    return (x - 0.5) / 0.5

def denorm_m11_to_01(x):
    return x * 0.5 + 0.5

def make_coord_grid(h, w, device, batch_size=1):
    y = torch.linspace(-1, 1, h, device=device)
    x = torch.linspace(-1, 1, w, device=device)
    yy, xx = torch.meshgrid(y, x, indexing="ij")
    coords = torch.stack([xx, yy], dim=-1).view(1, -1, 2)  # (1,N,2)
    cell = torch.tensor([2.0/h, 2.0/w], device=device, dtype=torch.float32).view(1, 2)
    if batch_size != 1:
        coords = coords.repeat(batch_size, 1, 1)
        cell = cell.repeat(batch_size, 1)
    return coords, cell

def rgb_to_y(img):  # img (...,3) in [0,1]
    return 0.2567 * img[...,0] + 0.5041 * img[...,1] + 0.0979 * img[...,2] + 16/255

@torch.no_grad()
def calc_psnr_y(pred01, target01, shave=2):
    pred01 = pred01.clamp(0,1)
    target01 = target01.clamp(0,1)

    pred01 = pred01[..., shave:-shave, shave:-shave]
    target01 = target01[..., shave:-shave, shave:-shave]

    pred = pred01.permute(0,2,3,1)
    tgt  = target01.permute(0,2,3,1)

    y_pred = rgb_to_y(pred)
    y_tgt  = rgb_to_y(tgt)

    mse = torch.mean((y_pred - y_tgt) ** 2).item()
    if not math.isfinite(mse) or mse <= 0:
        return float("nan") if not math.isfinite(mse) else 100.0
    return 20.0 * math.log10(1.0 / math.sqrt(mse))


# deterministic crop (file마다 항상 같은 좌표)
def deterministic_crop_xy(img_w, img_h, crop, key: str, seed=0):
    if img_w <= crop or img_h <= crop:
        return 0, 0
    s = f"{key}|{seed}".encode("utf-8")
    h = int(hashlib.md5(s).hexdigest(), 16)
    x = h % (img_w - crop + 1)
    y = (h // 1000003) % (img_h - crop + 1)
    return int(x), int(y)

def crop_pil(img, x, y, crop):
    return img.crop((x, y, x + crop, y + crop))

In [7]:
# ============================================================
# 4) Model definition (match srno_exp_base_final.py)
# ============================================================
class FourierPositionalEncoding(nn.Module):
    def __init__(self, L=5):
        super().__init__()
        self.L = L
        self.register_buffer("freq_bands", 2 ** torch.linspace(0, L-1, L), persistent=False)

    def forward(self, x):  # (B,N,2)
        pe = []
        for freq in self.freq_bands.to(x.device):
            pe.append(torch.sin(x * freq * math.pi))
            pe.append(torch.cos(x * freq * math.pi))
        return torch.cat(pe, dim=-1)  # (B,N, 2*L*2)

class GalerkinAttention(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        assert dim % heads == 0
        self.heads = heads
        self.head_dim = dim // heads

        self.to_q = nn.Linear(dim, dim, bias=False)
        self.to_k = nn.Linear(dim, dim, bias=False)
        self.to_v = nn.Linear(dim, dim, bias=False)

        self.ln_k = nn.LayerNorm(dim)
        self.ln_v = nn.LayerNorm(dim)

        self.to_out = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape

        q = self.to_q(x).reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)
        k = self.to_k(x).reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)
        v = self.to_v(x).reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)

        k2 = k.permute(0, 2, 1, 3).reshape(B, N, C)
        v2 = v.permute(0, 2, 1, 3).reshape(B, N, C)
        k2 = self.ln_k(k2)
        v2 = self.ln_v(v2)
        k = k2.reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)
        v = v2.reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)

        context = torch.matmul(k.transpose(-2, -1), v) / float(N)
        out = torch.matmul(q, context)

        out = out.permute(0, 2, 1, 3).reshape(B, N, C)
        return self.to_out(out)

class ResBlock(nn.Module):
    def __init__(self, n_feats, kernel_size=3, act=nn.ReLU(True), res_scale=0.1):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(n_feats, n_feats, kernel_size, padding=kernel_size//2),
            act,
            nn.Conv2d(n_feats, n_feats, kernel_size, padding=kernel_size//2),
        )
        self.res_scale = res_scale

    def forward(self, x):
        return x + self.body(x) * self.res_scale

class EDSR(nn.Module):
    def __init__(self, n_resblocks=16, n_feats=64):
        super().__init__()
        self.head = nn.Sequential(nn.Conv2d(3, n_feats, 3, padding=1))
        m_body = [ResBlock(n_feats, 3, res_scale=0.1) for _ in range(n_resblocks)]
        m_body.append(nn.Conv2d(n_feats, n_feats, 3, padding=1))
        self.body = nn.Sequential(*m_body)

    def forward(self, x):
        x = self.head(x)
        res = self.body(x)
        return x + res

class SRNO_Fourier(nn.Module):
    def __init__(self, residual_scale=0.1, use_fourier=False, L=5, width=128, blocks=8, corner_agg="concat"):
        super().__init__()
        assert corner_agg in ["concat", "inv_area"]
        self.corner_agg = corner_agg

        self.encoder = EDSR(16, 64)
        self.use_fourier = use_fourier
        self.pos_enc = FourierPositionalEncoding(L=L)

        corner_dim = 64 + 2 + 2
        if use_fourier:
            corner_dim += (2 * L * 2)

        in_dim = (4 * corner_dim) if (self.corner_agg == "concat") else corner_dim

        self.latent_dim = width
        self.lifting = nn.Linear(in_dim, self.latent_dim)

        self.layers = nn.ModuleList([GalerkinAttention(self.latent_dim, heads=8) for _ in range(blocks)])
        self.ffns = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(self.latent_dim),
                nn.Linear(self.latent_dim, self.latent_dim),
                nn.GELU(),
                nn.Linear(self.latent_dim, self.latent_dim),
            ) for _ in range(blocks)
        ])

        self.projection = nn.Linear(self.latent_dim, 3)
        self.residual_scale = residual_scale

    @staticmethod
    def _feat_coord_grid(Hf, Wf, device, B):
        y = torch.linspace(-1, 1, Hf, device=device, dtype=torch.float32)
        x = torch.linspace(-1, 1, Wf, device=device, dtype=torch.float32)
        yy, xx = torch.meshgrid(y, x, indexing="ij")
        grid = torch.stack([xx, yy], dim=-1).unsqueeze(0)
        return grid.repeat(B, 1, 1, 1)

    def forward(self, x, coords, cell):
        B = x.shape[0]
        coords = coords.to(dtype=torch.float32)
        cell   = cell.to(dtype=torch.float32)

        grid = coords.unsqueeze(2)

        base = F.grid_sample(x, grid, mode="bilinear", align_corners=False)
        base = base.squeeze(3).permute(0,2,1)

        feat = self.encoder(x)
        _, _, Hf, Wf = feat.shape
        feat_coord = self._feat_coord_grid(Hf, Wf, feat.device, B).permute(0,3,1,2)

        rx, ry = 1.0 / Hf, 1.0 / Wf
        eps = 1e-6
        vx_lst = [-1, 1]
        vy_lst = [-1, 1]

        rel_cell = cell.unsqueeze(1).repeat(1, coords.shape[1], 1).clone()
        rel_cell[..., 0] *= Hf
        rel_cell[..., 1] *= Wf

        preds = []
        areas = []
        for vx in vx_lst:
            for vy in vy_lst:
                coord_ = coords.clone()
                coord_[..., 0] = (coord_[..., 0] + vx * rx).clamp(-1 + eps, 1 - eps)
                coord_[..., 1] = (coord_[..., 1] + vy * ry).clamp(-1 + eps, 1 - eps)
                grid_ = coord_.unsqueeze(2)

                q_feat = F.grid_sample(feat, grid_, mode="nearest", align_corners=False).squeeze(3).permute(0,2,1)
                q_coord = F.grid_sample(feat_coord, grid_, mode="nearest", align_corners=False).squeeze(3).permute(0,2,1)

                rel_coord = coords - q_coord
                rel_coord[..., 0] *= Hf
                rel_coord[..., 1] *= Wf

                area = (rel_coord[..., 0].abs() * rel_coord[..., 1].abs())
                areas.append(area)

                parts = [q_feat, rel_coord]
                if self.use_fourier:
                    parts.append(self.pos_enc(rel_coord))
                parts.append(rel_cell)
                preds.append(torch.cat(parts, dim=-1))

        if self.corner_agg == "concat":
            z_in = torch.cat(preds, dim=-1)
        else:
            eps_w = 1e-9
            inv = [1.0 / (a + eps_w) for a in areas]
            inv_sum = (inv[0] + inv[1] + inv[2] + inv[3]).clamp_min(eps_w)
            w = [(iv / inv_sum).unsqueeze(-1) for iv in inv]
            z_in = preds[0]*w[0] + preds[1]*w[1] + preds[2]*w[2] + preds[3]*w[3]

        z = self.lifting(z_in)
        for attn, ffn in zip(self.layers, self.ffns):
            z = z + attn(z)
            z = z + ffn(z)

        residual = self.projection(z) * self.residual_scale
        return base + residual

In [8]:
# ============================================================
# 5) Load models from best_checkpoint.pth
# ============================================================
def build_model_from_config(cfg):
    return SRNO_Fourier(
        residual_scale=cfg.get("RESIDUAL_SCALE", 0.1),
        use_fourier=cfg.get("USE_FOURIER", False),
        L=cfg.get("L", 5),
        width=cfg.get("WIDTH", 128),
        blocks=cfg.get("BLOCKS", 8),
        corner_agg=cfg.get("CORNER_AGG", "concat"),
    ).to(DEVICE)

def load_best_model(exp_dir):
    ckpt_path = os.path.join(exp_dir, "best_checkpoint.pth")
    assert os.path.exists(ckpt_path), f"Missing: {ckpt_path}"
    ckpt = torch.load(ckpt_path, map_location="cpu")

    cfg = dict(ckpt["config"])
    model = build_model_from_config(cfg)
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    model.eval()
    return model, cfg

models = {}
cfgs = {}
for k, d in EXP_DIRS.items():
    m, cfg = load_best_model(d)
    models[k] = m
    cfgs[k] = cfg

print("loaded models:", list(models.keys()))
print("baseline cfg:", cfgs["baseline"])

✅ loaded models: ['baseline', 'fourier5', 'fourier10', 'inv_area']
baseline cfg: {'GPU_ID': 0, 'DATA_PATH': '/content/DIV2K', 'HR_SIZE': 192, 'SCALE_CHOICES_TRAIN': [2, 3, 4], 'SCALE_VAL': 2, 'BATCH_SIZE': 4, 'EPOCHS': 400, 'LEARNING_RATE': 4e-05, 'USE_AMP': True, 'WARMUP_EPOCHS': 20, 'WARMUP_MULTIPLIER': 3, 'TRAIN_COORD_SAMPLES': 4096, 'VAL_EVERY': 1, 'VIS_EVERY': 10, 'CKPT_SAVE_EVERY_STEPS': 500, 'DATA_NORM': True, 'EXP_NAME': 'SRNO_Base_final', 'USE_FOURIER': False, 'L': 5, 'CORNER_AGG': 'concat', 'WIDTH': 128, 'BLOCKS': 8, 'RESIDUAL_SCALE': 0.1}


In [9]:
# ============================================================
# 6) Eval one HR patch for a given scale
# ============================================================
@torch.no_grad()
def eval_one_patch_psnr(hr01, model, *, scale, data_norm=True):
    """
    hr01: (1,3,H,H) in [0,1]
    model expects input either [-1,1] (if data_norm True) or [0,1] (if False)
    """
    hr_size = hr01.shape[-1]
    lr_size = int(round(hr_size / float(scale)))
    lr_size = max(4, lr_size)

    lr01 = F.interpolate(hr01, size=(lr_size, lr_size), mode="bicubic", align_corners=False)

    if data_norm:
        hr_in = norm_01_to_m11(hr01)
        lr_in = norm_01_to_m11(lr01)
    else:
        hr_in = hr01
        lr_in = lr01

    coords, cell = make_coord_grid(hr_size, hr_size, DEVICE, batch_size=1)
    # IMPORTANT: scale-conditioned cell (match your comparison code)
    cell = cell * torch.tensor(scale, device=DEVICE, dtype=torch.float32).view(1, 1)

    pred = model(lr_in.to(DEVICE), coords, cell)  # (1,N,3)
    pred = pred.view(1, hr_size, hr_size, 3).permute(0,3,1,2).contiguous()

    if data_norm:
        pred01 = denorm_m11_to_01(pred)
    else:
        pred01 = pred

    psnr = calc_psnr_y(pred01, hr01.to(DEVICE), shave=2)
    return psnr

@torch.no_grad()
def eval_bicubic_psnr(hr01, *, scale):
    """
    hr01: (1,3,H,H) in [0,1]
    1) HR -> LR (bicubic down)
    2) LR -> SR (bicubic up)
    3) Y-PSNR vs HR
    """
    hr_size = hr01.shape[-1]
    lr_size = int(round(hr_size / float(scale)))
    lr_size = max(4, lr_size)

    lr01 = F.interpolate(hr01, size=(lr_size, lr_size), mode="bicubic", align_corners=False)
    bic01 = F.interpolate(lr01, size=(hr_size, hr_size), mode="bicubic", align_corners=False)

    return calc_psnr_y(bic01, hr01, shave=2)

In [10]:
# ============================================================
# 7) Main: average PSNR over 100 valid images (x2/x3/x4)
# ============================================================
# choose HR patch size: use training HR_SIZE=192 for consistency
HR_PATCH = 192

def load_hr_patch(path, hr_patch=192):
    img = Image.open(path).convert("RGB")
    W, H = img.size
    x, y = deterministic_crop_xy(W, H, hr_patch, key=os.path.basename(path), seed=SEED)
    hr = transforms.ToTensor()(crop_pil(img, x, y, hr_patch)).unsqueeze(0)  # (1,3,H,H) in [0,1]
    return hr, (x, y)

def compute_avg_psnr(scales=(2.0, 3.0, 4.0)):
    model_names = list(models.keys()) + ["bicubic"]
    results = {s: {name: [] for name in model_names} for s in scales}

    data_norm = bool(cfgs["baseline"].get("DATA_NORM", True))

    for path in tqdm(valid_files, desc="valid100"):
        hr01, _ = load_hr_patch(path, hr_patch=HR_PATCH)
        hr01 = hr01.to(DEVICE)

        for s in scales:
            results[s]["bicubic"].append(eval_bicubic_psnr(hr01, scale=float(s)))

            for name, model in models.items():
                dn = bool(cfgs[name].get("DATA_NORM", data_norm))
                ps = eval_one_patch_psnr(hr01, model, scale=float(s), data_norm=dn)
                results[s][name].append(ps)

    summary = {}
    for s in scales:
        summary[s] = {}
        for name, vals in results[s].items():
            arr = np.array(vals, dtype=np.float64)
            summary[s][name] = {
                "mean": float(np.nanmean(arr)),
                "std":  float(np.nanstd(arr)),
            }
    return summary

def compute_random_scale_avg_psnr(scale_choices=(2.0, 3.0, 4.0)):
    """
    각 이미지마다 scale을 랜덤으로 하나 선택해서
    평균 PSNR 계산
    """
    model_names = list(models.keys()) + ["bicubic"]
    results = {name: [] for name in model_names}

    data_norm_default = bool(cfgs["baseline"].get("DATA_NORM", True))

    for path in tqdm(valid_files, desc="valid100 (random-scale)"):
        hr01, _ = load_hr_patch(path, hr_patch=HR_PATCH)
        hr01 = hr01.to(DEVICE)

        s = float(random.choice(scale_choices))

        results["bicubic"].append(eval_bicubic_psnr(hr01, scale=s))

        for name, model in models.items():
            dn = bool(cfgs[name].get("DATA_NORM", data_norm_default))
            ps = eval_one_patch_psnr(hr01, model, scale=s, data_norm=dn)
            results[name].append(ps)

    summary = {}
    for name, vals in results.items():
        arr = np.array(vals, dtype=np.float64)
        summary[name] = {
            "mean": float(np.nanmean(arr)),
            "std":  float(np.nanstd(arr)),
        }
    return summary

In [11]:
summary = compute_avg_psnr(scales=(2.0, 3.0, 4.0))

valid100:   0%|          | 0/100 [00:00<?, ?it/s]

In [12]:
print("\n===== AVG Y-PSNR over DIV2K valid(100), HR_PATCH=192 =====")
for s in [2.0, 3.0, 4.0]:
    print(f"\n[scale x{s}]")
    for name in ["bicubic","baseline","fourier5","fourier10","inv_area"]:
        m = summary[s][name]["mean"]
        sd = summary[s][name]["std"]
        print(f"  {name:9s}: {m:.2f} ± {sd:.2f} dB")


===== AVG Y-PSNR over DIV2K valid(100), HR_PATCH=192 =====

[scale x2.0]
  bicubic  : 36.17 ± 10.69 dB
  baseline : 37.46 ± 9.06 dB
  fourier5 : 37.42 ± 8.90 dB
  fourier10: 37.46 ± 9.01 dB
  inv_area : 37.01 ± 9.12 dB

[scale x3.0]
  bicubic  : 32.50 ± 15.66 dB
  baseline : 33.05 ± 9.51 dB
  fourier5 : 33.05 ± 9.44 dB
  fourier10: 33.08 ± 9.56 dB
  inv_area : 32.81 ± 9.56 dB

[scale x4.0]
  bicubic  : 31.17 ± 16.35 dB
  baseline : 31.50 ± 9.81 dB
  fourier5 : 31.46 ± 9.63 dB
  fourier10: 31.51 ± 9.79 dB
  inv_area : 31.22 ± 9.79 dB


In [13]:
random_summary = compute_random_scale_avg_psnr((2.0,3.0,4.0))

valid100 (random-scale):   0%|          | 0/100 [00:00<?, ?it/s]

In [14]:
print("\n===== AVG Y-PSNR (Random scale per image) =====")
for name in ["bicubic","baseline","fourier5","fourier10","inv_area"]:
    m = random_summary[name]["mean"]
    sd = random_summary[name]["std"]
    print(f"{name:9s}: {m:.2f} ± {sd:.2f} dB")


===== AVG Y-PSNR (Random scale per image) =====
bicubic  : 33.79 ± 15.74 dB
baseline : 34.30 ± 9.74 dB
fourier5 : 34.28 ± 9.66 dB
fourier10: 34.32 ± 9.78 dB
inv_area : 33.98 ± 9.76 dB


##LNO

In [ ]:
# ============================================================
# 1) 경로 및 로컬 캐싱 (LNO 스타일)
# ============================================================
PROJECT_PATH = '/content/drive/MyDrive/super_solutioner/LNO_base'
DRIVE_DATA_PATH = os.path.join(PROJECT_PATH, 'data/DIV2K')
LOCAL_DATA_PATH = '/content/DIV2K'
os.makedirs(LOCAL_DATA_PATH, exist_ok=True)

def setup_div2k_from_drive(split="valid"):
    zip_name = f"DIV2K_{split}_HR.zip"
    drive_zip_path = os.path.join(DRIVE_DATA_PATH, zip_name)
    extracted_folder = os.path.join(LOCAL_DATA_PATH, f"DIV2K_{split}_HR")

    if not os.path.exists(drive_zip_path):
        print(f"❌ 에러: {drive_zip_path} 파일이 없습니다!")
        return

    if not os.path.exists(extracted_folder):
        print(f"[{split}] 🚀 드라이브에서 코랩 로컬로 압축 해제 중...")
        os.system(f"unzip -q {drive_zip_path} -d {LOCAL_DATA_PATH}")
        print(f"[{split}] 완료!")
    else:
        print(f"[{split}] ⚡ 로컬에 이미 준비됨.")

setup_div2k_from_drive("valid")

# ============================================================
# 2) 실험 경로 세팅 (CoDA-LNO 버전)
# ============================================================
EXP_DIRS = {
    "coda_lno": os.path.join(PROJECT_PATH, 'checkpoints_codalno_upgrade'),
    # "baseline": os.path.join(PROJECT_PATH, 'checkpoints_codalno_base'),
}

DIV2K_ROOT = LOCAL_DATA_PATH
VALID_GLOB = os.path.join(DIV2K_ROOT, "DIV2K_valid_HR", "*.png")
valid_files = sorted(glob.glob(VALID_GLOB))
assert len(valid_files) == 100, f"Expected 100 images, got {len(valid_files)}"
print("Valid images:", len(valid_files))

In [ ]:
# ============================================================
# 3) Utils (same as srno_exp_base_final.py / srno_comparison.py)
# ============================================================
def norm_01_to_m11(x):
    return (x - 0.5) / 0.5

def denorm_m11_to_01(x):
    return x * 0.5 + 0.5

def make_coord_grid(h, w, device, batch_size=1):
    y = torch.linspace(-1, 1, h, device=device)
    x = torch.linspace(-1, 1, w, device=device)
    yy, xx = torch.meshgrid(y, x, indexing="ij")
    coords = torch.stack([xx, yy], dim=-1).view(1, -1, 2)  # (1,N,2)
    cell = torch.tensor([2.0/h, 2.0/w], device=device, dtype=torch.float32).view(1, 2)
    if batch_size != 1:
        coords = coords.repeat(batch_size, 1, 1)
        cell = cell.repeat(batch_size, 1)
    return coords, cell

def rgb_to_y(img):  # img (...,3) in [0,1]
    return 0.2567 * img[...,0] + 0.5041 * img[...,1] + 0.0979 * img[...,2] + 16/255

@torch.no_grad()
def calc_psnr_y(pred01, target01, shave=2):
    pred01 = pred01.clamp(0,1)
    target01 = target01.clamp(0,1)

    pred01 = pred01[..., shave:-shave, shave:-shave]
    target01 = target01[..., shave:-shave, shave:-shave]

    pred = pred01.permute(0,2,3,1)
    tgt  = target01.permute(0,2,3,1)

    y_pred = rgb_to_y(pred)
    y_tgt  = rgb_to_y(tgt)

    mse = torch.mean((y_pred - y_tgt) ** 2).item()
    if not math.isfinite(mse) or mse <= 0:
        return float("nan") if not math.isfinite(mse) else 100.0
    return 20.0 * math.log10(1.0 / math.sqrt(mse))


# deterministic crop (file마다 항상 같은 좌표)
def deterministic_crop_xy(img_w, img_h, crop, key: str, seed=0):
    if img_w <= crop or img_h <= crop:
        return 0, 0
    s = f"{key}|{seed}".encode("utf-8")
    h = int(hashlib.md5(s).hexdigest(), 16)
    x = h % (img_w - crop + 1)
    y = (h // 1000003) % (img_h - crop + 1)
    return int(x), int(y)

def crop_pil(img, x, y, crop):
    return img.crop((x, y, x + crop, y + crop))

In [ ]:
# ============================================================
# 4. Model Definitions (LNO Baseline & CoDA-LNO)
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# ------------------------------------------------------------
# [A] LNO Baseline (DeepLNO_SR) 관련 클래스
# ------------------------------------------------------------
class PR2d_SR(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super(PR2d_SR, self).__init__()
        self.modes1 = modes1
        self.modes2 = modes2
        self.scale = (1 / (in_channels * out_channels))

        self.weights_pole1 = nn.Parameter(self.scale * torch.randn(in_channels, out_channels, self.modes1, dtype=torch.cfloat))
        self.weights_pole2 = nn.Parameter(self.scale * torch.randn(in_channels, out_channels, self.modes2, dtype=torch.cfloat))
        self.weights_residue = nn.Parameter(self.scale * torch.randn(in_channels, out_channels, self.modes1, self.modes2, dtype=torch.cfloat))

    def output_PR(self, lambda1, lambda2, alpha, weights_pole1, weights_pole2, weights_residue):
        term1 = lambda1 - weights_pole1
        term2 = lambda2 - weights_pole2
        denom = term1.unsqueeze(-1) * term2.unsqueeze(-2)

        epsilon = 1e-5
        denom = torch.where(torch.abs(denom) < epsilon, denom + epsilon, denom)

        H = torch.div(weights_residue, denom)
        out_residue = torch.einsum("bixy,ioxy->boxy", alpha, H)
        return out_residue

    def forward(self, x, target_size):
        B, C, H, W = x.shape
        alpha = torch.fft.fft2(x, dim=[-2, -1])
        alpha_modes = alpha[..., :self.modes1, :self.modes2]

        omega1 = torch.fft.fftfreq(self.modes1, d=1/self.modes1).to(x.device) * 2 * np.pi * 1j
        omega2 = torch.fft.fftfreq(self.modes2, d=1/self.modes2).to(x.device) * 2 * np.pi * 1j
        lambda1 = omega1.reshape(1, 1, self.modes1)
        lambda2 = omega2.reshape(1, 1, self.modes2)

        out_res = self.output_PR(lambda1, lambda2, alpha_modes,
                                 self.weights_pole1, self.weights_pole2, self.weights_residue)

        x_out = torch.fft.ifft2(out_res, s=target_size, dim=[-2, -1])
        return torch.real(x_out)

class LNOBlock(nn.Module):
    def __init__(self, width, modes1, modes2):
        super().__init__()
        self.norm1 = nn.InstanceNorm2d(width)
        self.lno = PR2d_SR(width, width, modes1, modes2)
        self.act = nn.GELU()

        self.norm2 = nn.InstanceNorm2d(width)
        self.mlp = nn.Sequential(
            nn.Conv2d(width, width * 2, 1),
            nn.GELU(),
            nn.Conv2d(width * 2, width, 1)
        )

    def forward(self, x):
        target_size = (x.shape[2], x.shape[3])
        resid = self.lno(self.norm1(x), target_size)
        x = x + self.act(resid)

        resid = self.mlp(self.norm2(x))
        x = x + resid
        return x

class DeepLNO_SR(nn.Module):
    def __init__(self, in_channels=3, width=64, layers=4, modes1=32, modes2=32):
        super(DeepLNO_SR, self).__init__()
        self.lifting = nn.Conv2d(in_channels, width, 1)
        self.layers = nn.ModuleList([
            LNOBlock(width, modes1, modes2) for _ in range(layers)
        ])
        self.upsampler = PR2d_SR(width, width, modes1, modes2)
        self.proj = nn.Conv2d(width, in_channels, 1)

    def forward(self, x, target_size=None, scale_factor=None):
        # (호환성을 위해 scale_factor 인자 추가: 내부적으로 안 써도 에러 방지용)
        if target_size is None:
            target_size = (x.shape[2]*4, x.shape[3]*4)

        x_feat = self.lifting(x)
        for layer in self.layers:
            x_feat = layer(x_feat)
        x_up = self.upsampler(x_feat, target_size=target_size)
        out = self.proj(x_up)
        base = F.interpolate(x, size=target_size, mode='bicubic', align_corners=False)
        return out + base

# ------------------------------------------------------------
# [B] CoDA-LNO (CoDALNO_SR) 관련 클래스
# ------------------------------------------------------------
class LaplaceSpatialMixer(nn.Module):
    def __init__(self, dim, modes1=16, modes2=16):
        super().__init__()
        self.modes1 = modes1
        self.modes2 = modes2
        self.scale = (1 / (dim * dim))

        self.weights_pole1 = nn.Parameter(self.scale * torch.rand(1, 1, modes1, dtype=torch.cfloat))
        self.weights_pole2 = nn.Parameter(self.scale * torch.rand(1, 1, modes2, dtype=torch.cfloat))
        self.weights_residue = nn.Parameter(self.scale * torch.rand(dim, dim, modes1, modes2, dtype=torch.cfloat))

    def forward(self, x, target_size=None):
        B, C, H, W = x.shape
        if target_size is None: target_size = (H, W)

        alpha = torch.fft.fft2(x, dim=[-2, -1])
        alpha_modes = alpha[..., :self.modes1, :self.modes2]

        omega1 = torch.fft.fftfreq(self.modes1, d=1/self.modes1).to(x.device) * 2 * np.pi * 1j
        omega2 = torch.fft.fftfreq(self.modes2, d=1/self.modes2).to(x.device) * 2 * np.pi * 1j
        lambda1 = omega1.reshape(1, 1, self.modes1)
        lambda2 = omega2.reshape(1, 1, self.modes2)

        term1 = lambda1 - self.weights_pole1
        term2 = lambda2 - self.weights_pole2
        denom = term1.unsqueeze(-1) * term2.unsqueeze(-2)

        H_s = torch.div(self.weights_residue, denom)
        out_freq = torch.einsum("bixy,ioxy->boxy", alpha_modes, H_s)
        out = torch.fft.ifft2(out_freq, s=target_size, dim=[-2, -1])
        return torch.real(out)

class ChannelMixer(nn.Module):
    def __init__(self, dim, expansion=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(dim, dim * expansion, 1),
            nn.GELU(),
            nn.Conv2d(dim * expansion, dim, 1)
        )
    def forward(self, x):
        return self.net(x)

class CoDALNOBlock(nn.Module):
    def __init__(self, dim, modes=16, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.InstanceNorm2d(dim)
        self.spatial_mixer = LaplaceSpatialMixer(dim, modes1=modes, modes2=modes)
        self.norm2 = nn.InstanceNorm2d(dim)
        self.channel_mixer = ChannelMixer(dim, expansion=mlp_ratio)
        self.alpha = nn.Parameter(torch.ones(1, dim, 1, 1))
        self.beta = nn.Parameter(torch.ones(1, dim, 1, 1))

    def forward(self, x):
        resid = self.spatial_mixer(self.norm1(x))
        x = x + self.alpha * resid
        resid = self.channel_mixer(self.norm2(x))
        x = x + self.beta * resid
        return x

class CoDALNO_SR(nn.Module):
    def __init__(self, in_channels=3, width=64, blocks=4, modes=16):
        super().__init__()
        self.width = width
        self.lifting = nn.Conv2d(in_channels, width, 1)

        self.scale_embed = nn.Sequential(
            nn.Linear(1, width),
            nn.GELU(),
            nn.Linear(width, width)
        )

        self.blocks = nn.ModuleList([
            CoDALNOBlock(width, modes=modes) for _ in range(blocks)
        ])

        self.upsampler = LaplaceSpatialMixer(width, modes1=modes, modes2=modes)
        self.proj = nn.Conv2d(width, in_channels, 1)

    def forward(self, x, target_size, scale_factor):
        if isinstance(scale_factor, (int, float)):
            scale_factor = torch.full((x.shape[0], 1), scale_factor, dtype=torch.float32, device=x.device)
        elif scale_factor.dim() == 1:
            scale_factor = scale_factor.unsqueeze(1)

        x_feat = self.lifting(x)
        s_emb = self.scale_embed(scale_factor).view(-1, self.width, 1, 1)
        x_feat = x_feat + s_emb

        shortcut = x_feat
        for block in self.blocks:
            x_feat = block(x_feat)
        x_feat = x_feat + shortcut

        x_up = self.upsampler(x_feat, target_size=target_size)
        out = self.proj(x_up)
        base = F.interpolate(x, size=target_size, mode='bicubic', align_corners=False)
        return out + base

In [ ]:
# ============================================================
# 5) 모델 로드 (LNO vs CoDA-LNO vs Upgrade 3종 세트 비교)
# ============================================================
import os

PROJECT_PATH = '/content/drive/MyDrive/super_solutioner/LNO_base'

EXP_DIRS = {
    "lno_base": os.path.join(PROJECT_PATH, "checkpoints", "epoch_2000.pth"),
    "codalno":  os.path.join(PROJECT_PATH, "checkpoints_codalno", "codalno_epoch_2000.pth"), # 여기서 에러 안 나게 세팅!
    "upgrade":  os.path.join(PROJECT_PATH, "checkpoints_codalno_upgrade", "codalno_latest.pth")
}

def load_eval_model(name, ckpt_path):
    if not os.path.exists(ckpt_path):
        print(f"⚠️ 파일 없음: {ckpt_path}")
        return None

    checkpoint = torch.load(ckpt_path, map_location="cpu")

    # ----------------------------------------------------
    # 각 모델별 학습 당시의 파라미터(width, layers/blocks) 명시
    # ----------------------------------------------------
    if name == "lno_base":
        # DeepLNO 모델: width=64, layers=4
        model = DeepLNO_SR(in_channels=3, width=64, layers=4, modes1=32, modes2=32).to(DEVICE)

    elif name == "codalno":
        # 첫 번째 CoDA-LNO 모델: width=64, blocks=6
        model = CoDALNO_SR(in_channels=3, width=64, blocks=6, modes=32).to(DEVICE)

    elif name == "upgrade":
        # 업그레이드된 CoDA-LNO 모델: width=128, blocks=10
        model = CoDALNO_SR(in_channels=3, width=128, blocks=10, modes=32).to(DEVICE)

    # ----------------------------------------------------
    # State dict 불러오기
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'], strict=True)
    else:
        model.load_state_dict(checkpoint, strict=True)

    model.eval()
    return model

models = {}
for name, ckpt_path in EXP_DIRS.items():
    print(f"🔄 로드 중: {name:10s} <- {os.path.basename(ckpt_path)}")
    m = load_eval_model(name, ckpt_path)
    if m is not None:
        models[name] = m

print("준비된 모델:", list(models.keys()))

In [ ]:
# ============================================================
# 6) Eval one HR patch for a given scale (SRNO & LNO 호환 평가 모드)
# ============================================================
@torch.no_grad()
def eval_one_patch_psnr(hr01, model, *, scale, model_type="lno"):
    hr_size = hr01.shape[-1]
    lr_size = int(round(hr_size / float(scale)))
    lr_size = max(4, lr_size)

    # 파이토치 기반 Bicubic 통일 (입력 문제지 통일)
    lr01 = F.interpolate(hr01, size=(lr_size, lr_size), mode="bicubic", align_corners=False)

    if model_type == "srno":
        # 기존 SRNO 모델을 위한 로직
        hr_in = norm_01_to_m11(hr01)
        lr_in = norm_01_to_m11(lr01)
        coords, cell = make_coord_grid(hr_size, hr_size, DEVICE, batch_size=1)
        cell = cell * torch.tensor(scale, device=DEVICE, dtype=torch.float32).view(1, 1)

        pred = model(lr_in.to(DEVICE), coords, cell)
        pred = pred.view(1, hr_size, hr_size, 3).permute(0,3,1,2).contiguous()
        pred01 = denorm_m11_to_01(pred)

    elif model_type == "lno":
        # LNO & CoDA-LNO 모델을 위한 로직 (에러 해결!)
        target_size = (hr_size, hr_size)
        # coords, cell 대신 target_size와 scale_factor 전달
        pred01 = model(lr01.to(DEVICE), target_size=target_size, scale_factor=float(scale))
        pred01 = torch.clamp(pred01, 0.0, 1.0)

    # 채점 기준 통일: 가장자리 2픽셀(shave=2) 무시하고 계산
    psnr = calc_psnr_y(pred01, hr01.to(DEVICE), shave=2)
    return psnr
# 기존 메모리에 살아있는 함수
HR_PATCH = 192

def load_hr_patch(path, hr_patch=192):
    img = Image.open(path).convert("RGB")
    W, H = img.size
    # 모든 모델이 똑같은 위치를 자르도록 해시(Hash) 기반의 고정 좌표 생성
    x, y = deterministic_crop_xy(W, H, hr_patch, key=os.path.basename(path), seed=SEED)
    hr = transforms.ToTensor()(crop_pil(img, x, y, hr_patch)).unsqueeze(0)  # (1,3,H,H) in [0,1]
    return hr, (x, y)
# ============================================================
# 7) Main: 평가 실행 루프 (고정 배율 & 랜덤 배율)
# ============================================================
def compute_avg_psnr(scales=(2.0, 3.0, 4.0)):
    results = {s: {name: [] for name in models.keys()} for s in scales}

    for path in tqdm(valid_files, desc="valid100"):
        hr01, _ = load_hr_patch(path, hr_patch=192)
        hr01 = hr01.to(DEVICE)

        for s in scales:
            for name, model in models.items():
                # 이름에 'lno'나 'upgrade'가 들어가면 lno 모드, 아니면 srno 모드
                m_type = "lno" if "lno" in name.lower() or "upgrade" in name.lower() else "srno"

                ps = eval_one_patch_psnr(hr01, model, scale=float(s), model_type=m_type)
                results[s][name].append(ps)

    summary = {}
    for s in scales:
        summary[s] = {}
        for name, vals in results[s].items():
            arr = np.array(vals, dtype=np.float64)
            summary[s][name] = {
                "mean": float(np.nanmean(arr)),
                "std":  float(np.nanstd(arr)),
            }
    return summary

def compute_random_scale_avg_psnr(scale_choices=(2.0, 3.0, 4.0)):
    results = {name: [] for name in models.keys()}

    for path in tqdm(valid_files, desc="valid100 (random-scale)"):
        hr01, _ = load_hr_patch(path, hr_patch=192)
        hr01 = hr01.to(DEVICE)

        s = float(random.choice(scale_choices))

        for name, model in models.items():
            m_type = "lno" if "lno" in name.lower() or "upgrade" in name.lower() else "srno"
            ps = eval_one_patch_psnr(hr01, model, scale=s, model_type=m_type)
            results[name].append(ps)

    summary = {}
    for name, vals in results.items():
        arr = np.array(vals, dtype=np.float64)
        summary[name] = {
            "mean": float(np.nanmean(arr)),
            "std":  float(np.nanstd(arr)),
        }
    return summary

In [ ]:
summary = compute_avg_psnr(scales=(2.0, 3.0, 4.0))

In [ ]:
# ============================================================
# 출력 1: 고정 스케일 평균 PSNR
# ============================================================
print("\n===== AVG Y-PSNR over DIV2K valid(100), HR_PATCH=192 =====")
for s in [2.0, 3.0, 4.0]:
    print(f"\n[scale x{s}]")
    # 하드코딩 대신 summary에 있는 키(모델 이름)를 자동으로 순회
    for name in models.keys():
        if name in summary[s]:
            m = summary[s][name]["mean"]
            sd = summary[s][name]["std"]
            print(f"  {name:9s}: {m:.2f} ± {sd:.2f} dB")

In [ ]:
random_summary = compute_random_scale_avg_psnr((2.0,3.0,4.0))

In [ ]:
# ============================================================
# 출력 2: 랜덤 스케일 평균 PSNR
# ============================================================
print("\n===== AVG Y-PSNR (Random scale per image) =====")
for name in models.keys():
    if name in random_summary:
        m = random_summary[name]["mean"]
        sd = random_summary[name]["std"]
        print(f"  {name:9s}: {m:.2f} ± {sd:.2f} dB")